# Fase 4: fine-tuning QLoRA del generador (LetraDeMédico)

Ajusta **Qwen2.5-3B-Instruct** con QLoRA sobre las respuestas de los profesores (Qwen3.8-27B y gpt-oss-120b) que pasaron las verificaciones automáticas y el guardrail de la Fase 3. Después genera las respuestas del test con el modelo **base** y con el **ajustado**, para compararlos en `evaluacion/evaluar_generador.py`.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

Todo lo importante se guarda en **Google Drive** (`MyDrive/letrademedico`): si Colab se desconecta, vuelve a correr las celdas 1 a 3 y luego la que se cortó; el entrenamiento y la generación continúan donde quedaron.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/letrademedico'
ADAPTADOR = f'{DRIVE}/qwen2.5-3b-instruct-lora'
!mkdir -p {DRIVE}/checkpoints {DRIVE}/salidas

In [ ]:
%cd /content
!test -d Proyecto_LaLetraDeMedico || git clone https://github.com/Valero25/Proyecto_LaLetraDeMedico.git
%cd /content/Proyecto_LaLetraDeMedico
!git pull -q && git log --oneline -1
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets
# Las salidas del test van directo a Drive, para no perderlas si se corta la sesión
!rm -rf data/generador/salidas && ln -s {DRIVE}/salidas data/generador/salidas
!wc -l data/generador/sft_train.jsonl data/generador/sft_val.jsonl

## Entrenamiento

Imprime el conteo de parámetros entrenables por tipo de capa (r·(d + k) frente a d·k) y la pérdida cada 5 pasos; evalúa en validación cada 25. Al final guarda en Drive el adaptador LoRA y `entrenamiento.json` con las curvas de pérdida. En la T4 tarda del orden de 2 horas.

In [ ]:
!python training/generador/ajustar_qlora.py --checkpoints {DRIVE}/checkpoints --salida {ADAPTADOR}

## Respuestas del test (150 casos)

Primero el modelo **base** (sin ajustar) y después el **ajustado**, con el mismo prompt y decodificación voraz. Cada una tarda del orden de 15 a 30 minutos.

In [ ]:
!python training/generador/generar_respuestas.py --nombre qwen2.5-3b-base

In [ ]:
!python training/generador/generar_respuestas.py --adaptador {ADAPTADOR} --nombre qwen2.5-3b-ajustado

## Descargar resultados

El zip trae el adaptador LoRA (para la app, Fase 5), `entrenamiento.json` y las salidas del test. Descomprímelo en la raíz del proyecto local.

In [ ]:
!rm -rf /content/fase4 && mkdir -p /content/fase4/models/generador /content/fase4/data/generador/salidas
!cp -r {ADAPTADOR} /content/fase4/models/generador/
!cp {DRIVE}/salidas/*.jsonl /content/fase4/data/generador/salidas/
%cd /content/fase4
!zip -qr /content/fase4_qlora.zip models data && ls -lh /content/fase4_qlora.zip
%cd /content/Proyecto_LaLetraDeMedico
from google.colab import files
files.download('/content/fase4_qlora.zip')